In [3]:
import os
import json
import pickle
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [5]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELS     = CONFIGS['experiments']
SPLIT      = 'test'
FIELDVARS  = MODELS['sr']['runs']['sr_gauss']['fieldvars']
NNSEEDS    = MODELS['nn']['seeds']
SRCONFIG   = MODELS['sr']
PODRUNS    = MODELS['pod']['runs']
MINSAMPLES = 30
NBINS      = 40

COLORS = {}
LABELS = {}
for name,config in SRCONFIG['optimizedeqs'].items():
    COLORS[name] = config['color']
    LABELS[name] = config['description']
for name,config in PODRUNS.items():
    COLORS[name] = config['color']
    LABELS[name] = config['description']

In [6]:
def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None: w = w*mask[:,None,:]
    return w.sum(axis=2)

def to_physical(norm):
    return np.expm1(tpstd*np.maximum(0.0,np.asarray(norm,dtype=float)))

def bin_1d(x,z,nbins=NBINS,minsamples=MINSAMPLES,plo=1,phi=99):
    finite = np.isfinite(x)&np.isfinite(z)
    x,z    = x[finite],z[finite]
    edges  = np.unique(np.percentile(x,np.linspace(plo,phi,nbins+1)))
    n      = len(edges)-1
    xi     = np.clip(np.digitize(x,edges)-1,0,n-1)
    counts = np.bincount(xi,minlength=n)
    sums   = np.bincount(xi,weights=z,minlength=n)
    return 0.5*(edges[:-1]+edges[1:]),np.where(counts>=minsamples,sums/counts,np.nan),counts

def bin_2d(x,y,z,xedges,yedges,minsamples=MINSAMPLES):
    finite = np.isfinite(x)&np.isfinite(y)&np.isfinite(z)
    x,y,z  = x[finite],y[finite],z[finite]
    nx,ny  = len(xedges)-1,len(yedges)-1
    xi     = np.clip(np.digitize(x,xedges)-1,0,nx-1)
    yi     = np.clip(np.digitize(y,yedges)-1,0,ny-1)
    idx    = yi*nx+xi
    counts = np.bincount(idx,minlength=nx*ny).reshape(ny,nx)
    sums   = np.bincount(idx,weights=z,minlength=nx*ny).reshape(ny,nx)
    return np.where(counts>=minsamp,sums/counts,np.nan),counts

def percentile_range(arr,lo=1,hi=99):
    a = arr[np.isfinite(arr)]
    return float(np.percentile(a,lo)),float(np.percentile(a,hi))

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
tpmean = float(STATS['tp_mean'])
tpstd  = float(STATS['tp_std'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nlat  = ds.sizes['lat']
    nlon  = ds.sizes['lon']
    nsig  = ds.sizes.get('sig',1)
    shape = (ntime,nlat,nlon)
    lats = ds.lat.values
    lons = ds.lon.values
    dsig = ds['dsig'].values
    levmin,levmax = CONFIGS['domain']['levrange']
    sigcoords = (ds.coords['sig'].values if 'sig' in ds.coords else np.linspace(levmin/1000,levmax/1000,nsig))
    farrs     = [ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS]
    fieldstack = np.stack(farrs,axis=1)
    surfmask   = (ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None)
    def getflat(da):
        if 'time' in da.dims: return da.transpose('time','lat','lon').values.ravel()
        return np.tile(da.values,(ntime,1,1)).ravel()
    blnorm  = getflat(ds['bl']) if 'bl' in ds else None
    lfnorm  = getflat(ds['lf'])  if 'lf'  in ds else None
    shfnorm = getflat(ds['shf']) if 'shf' in ds else None

kwlist = []
for seed in NNSEEDS:
    wpath = os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc')
    if os.path.exists(wpath):
        with xr.open_dataset(wpath,engine='h5netcdf') as wds:
            kwlist.append(wds['k'].values)
            if 'sig' in wds.coords and len(kwlist)==1:
                sigcoords = wds.coords['sig'].values
ki = (np.mean([kernel_integrate(fieldstack,kw,dsig,surfmask) for kw in kwlist],axis=0)
      if kwlist else fieldstack.mean(axis=2))
rhk,thetaek,thetaestark = ki[:,0],ki[:,1],ki[:,2]

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    trueflat = ds.tp.transpose('time','lat','lon').values.ravel()
    if 'lf' in ds:
        lf2d     = ds['lf'].squeeze() if 'time' not in ds['lf'].dims else ds['lf'].isel(time=0)
        landflat = np.broadcast_to((lf2d.values>=0.5)[None],refshape).ravel().copy()
    else:
        landflat = None

regpath = os.path.join(MODELSDIR,'sr','optimized_equations.pkl')
SR_REGISTRY = pickle.load(open(regpath,'rb')) if os.path.exists(regpath) else {}

valid = (np.isfinite(trueflat)&np.isfinite(rhk)
         &np.isfinite(thetaek)&np.isfinite(thetaestark))
print(f'Valid samples: {valid.sum():,}')

In [2]:
# ── SR-LO: binned by RH ──────────────────────────────────────────────────
rh_cen,rhnobsmeans,_ = bin1d(rhk[valid],trueflat[valid],plo=0,phi=100)
if 'sr_lo' in SR_REGISTRY:
    c      = SR_REGISTRY['sr_lo']['constants']
    rhnsw  = np.linspace(rh_cen[0],rh_cen[-1],300)
    Plotnat = to_phys(c['a']*np.exp(rhnsw)+c['b'])

# ── SR-BL: binned by BL ──────────────────────────────────────────────────
if blnorm is not None:
    bl_cen,blnobsmeans,_ = bin1d(blnorm[valid],trueflat[valid],plo=0,phi=100)
    if 'sr_bl' in SR_REGISTRY:
        c      = SR_REGISTRY['sr_bl']['constants']
        blnsw  = np.linspace(bl_cen[0],bl_cen[-1],300)
        Psrbl  = to_phys((blnsw+c['a'])**3+c['b'])

# ── POD-BL from predictions ──────────────────────────────────────────────
podpred = None
podpath = os.path.join(PREDSDIR,f'pod_bl_{SPLIT}_predictions.nc')
if os.path.exists(podpath):
    with xr.open_dataset(podpath,engine='h5netcdf') as ds:
        podpred = ds['tp'].transpose('time','lat','lon').values.ravel()
if podpred is not None and blnorm is not None:
    _,podblmeans,_ = bin1d(blnorm[valid],podpred[valid],plo=0,phi=100)

NameError: name 'bin1d' is not defined

In [ ]:
fig,axs = pplt.subplots(ncols=2,figwidth=4,sharex=False,sharey=True)
axs[0].scatter(rh_cen,rhnobsmeans,color='k',marker='.',s=20,zorder=0)
if 'sr_lo' in SR_REGISTRY:
    axs[0].plot(rhnsw,Plotnat,color=COLORS['sr_lo'],lw=1.5,zorder=5,label=LABELS['sr_lo'])
axs[0].axhline(0,color='gray',lw=0.5)
axs[0].format(title='SR-LO',xlabel=r'$\widehat{\mathrm{RH}}$ (%)',xlim=(-5,105),xticks=25)
axs[0].legend(loc='ul',ncols=1)

if blnorm is not None:
    finite = np.isfinite(blnobsmeans)
    axs[1].scatter(bl_cen[finite],blnobsmeans[finite],color='k',marker='.',s=20,zorder=0)
    if 'sr_bl' in SR_REGISTRY:
        axs[1].plot(blnsw,Psrbl,color=COLORS['sr_bl'],lw=1.5,zorder=5,label=LABELS['sr_bl'])
    if podpred is not None:
        axs[1].plot(bl_cen,podblmeans,color=COLORS['pod_bl'],lw=1.5,
                    zorder=4,label=LABELS['pod_bl'])
    axs[1].axhline(0,color='gray',lw=0.5)
    axs[1].format(title='SR-BL',xlabel=r'$\mathit{B_L}$ (m s$^{-2}$)',
                  xlim=(-0.55,0.05),xticks=0.25)
    axs[1].legend(loc='ul',ncols=1)
axs.format(abc=True,grid=False,ylabel='Total Precipitation (mm)',
           ylim=(-1,16),yticks=5,titleloc='l')
pplt.show()
fig.save('../figs/fig_3.jpg')

In [ ]:
if 'sr_med' not in SR_REGISTRY:
    print('sr_med not in registry')
else:
    c = SR_REGISTRY['sr_med']['constants']
    am,bm,cm = c['a'],c['b'],c['c']
    Mall = rhk[valid]; Iall = thetaek[valid]-bm*thetaestark[valid]-cm
    Pall = trueflat[valid]
    moist = Mall>=Iall; insta = ~moist
    print(f'Moisture-ctrl (M\u2265I): {100*moist.mean():.1f}%  mean P={Pall[moist].mean():.3f} mm')
    print(f'Instability   (I>M):    {100*insta.mean():.1f}%  mean P={Pall[insta].mean():.3f} mm')
    if landflat is not None:
        lv = landflat[valid].astype(bool)
        for lbl,lm in [('Land',lv),('Ocean',~lv)]:
            for rlbl,rm in [('Moisture',moist),('Instability',insta)]:
                mm = lm&rm
                print(f'  {lbl} {rlbl}: {100*mm.sum()/lm.sum():.1f}%  mean P={Pall[mm].mean():.3f}')

In [ ]:
if 'sr_med' in SR_REGISTRY:
    c = SR_REGISTRY['sr_med']['constants']
    am,bm,cm = c['a'],c['b'],c['c']
    M = rhk[valid]; I = thetaek[valid]-bm*thetaestark[valid]-cm; P = trueflat[valid]
    Mlo,Mhi = prange(M); Ilo,Ihi = prange(I)
    axlo = min(Mlo,Ilo); axhi = max(Mhi,Ihi)
    NB = 35
    xe = np.linspace(Mlo,Mhi,NB+1); ye = np.linspace(Ilo,Ihi,NB+1)
    xcen = 0.5*(xe[:-1]+xe[1:]); ycen = 0.5*(ye[:-1]+ye[1:])
    obsbin,_ = bin2d(M,I,P,xe,ye)
    srbin,_  = bin2d(M,I,to_phys(am*np.maximum(M,I)**3),xe,ye)
    density  = np.log10(np.histogram2d(M,I,bins=[xe,ye])[0].T.clip(1))
    NG = 200
    Mg,Ig    = np.meshgrid(np.linspace(Mlo,Mhi,NG),np.linspace(Ilo,Ihi,NG))
    Pgrid    = to_phys(am*np.maximum(Mg,Ig)**3)
    Mfull    = rhk.reshape(refshape)
    Ifull    = (thetaek-bm*thetaestark-cm).reshape(refshape)
    frac2d   = (Mfull>=Ifull).mean(axis=0)
    latlim   = (float(lats.min()),float(lats.max()))
    lonlim   = (float(lons.min()),float(lons.max()))
    xlabelM  = r'$\mathit{M}$ (Standardized Anomaly)'
    ylabelI  = r'$\mathit{I}$ (Standardized Anomaly)'
    kw    = dict(cmap='ColdHot_r',cmap_kw={'left':0.5},vmin=0,vmax=4,levels=18,extend='max')
    kwmap = dict(coast=True,lonlim=lonlim,lonlines=[65,75,85],lonlabels='b',
                 latlim=latlim,latlines=[10,15,20],latlabels='l',grid=False)
    fig,axs = pplt.subplots([[1,2],[3,4]],figwidth=5.5,proj={4:'cyl'},share=False,hspace=8)
    m0 = axs[0].pcolormesh(np.linspace(Mlo,Mhi,NG),np.linspace(Ilo,Ihi,NG),Pgrid,**kw)
    axs[0].plot([axlo,axhi],[axlo,axhi],'k--',lw=1)
    axs[0].text(Mhi-0.13*(Mhi-Mlo),Ilo+0.08*(Ihi-Ilo),'moisture-dominated',ha='right',va='bottom')
    axs[0].text(Mlo+0.05*(Mhi-Mlo),Ihi-0.12*(Ihi-Ilo),'instability-dominated',ha='left',va='top')
    axs[0].format(xlabel=xlabelM,ylabel=ylabelI,title='SR-MED',xlim=(Mlo,Mhi),ylim=(Ilo,Ihi))
    axs[1].pcolormesh(xcen,ycen,obsbin,**kw)
    axs[1].plot([axlo,axhi],[axlo,axhi],'k--',lw=1)
    axs[1].format(xlabel=xlabelM,ylabel='',yticklabels=[],title='ERA5',
                  xlim=(Mlo,Mhi),ylim=(Ilo,Ihi),facecolor='gray1')
    m2 = axs[2].pcolormesh(xcen,ycen,density,cmap='Grays',vmin=0)
    axs[2].plot([axlo,axhi],[axlo,axhi],'k--',lw=0.8)
    axs[2].format(xlabel=xlabelM,ylabel=ylabelI,title='Sample Density',
                  xlim=(Mlo,Mhi),ylim=(Ilo,Ihi))
    axs[2].colorbar(m2,loc='b',label=r'log$_{10}$($\mathit{N}$)')
    m3 = axs[3].pcolormesh(lons,lats,frac2d,cmap='ColdHot_r',vmin=0,vmax=1)
    axs[3].format(title='Moisture-Dominated Fraction',**kwmap)
    axs[3].colorbar(m3,loc='b',ticks=0.2,
                    label=r'Fraction ($\mathit{M}$ \u2265 $\mathit{I}$)')
    fig.format(abc=True,titleloc='l')
    fig.canvas.draw()
    figh   = fig.get_figheight()
    left   = min(axs[0].get_position().x0,axs[1].get_position().x0)
    right  = max(axs[0].get_position().x1,axs[1].get_position().x1)
    btop   = min(axs[0].get_position().y0,axs[1].get_position().y0)
    tbot   = max(axs[2].get_position().y1,axs[3].get_position().y1)
    cax    = fig.add_axes([left, tbot+0.5*(btop-tbot)-0.09/figh, right-left, 0.18/figh])
    cb     = fig.colorbar(m0,cax=cax,orientation='horizontal',
                          label='Total Precipitation (mm)',extend='max')
    cb.ax.tick_params(labelsize=9)
    pplt.show()
    fig.save('../figs/fig_4.jpg')

In [ ]:
if 'sr_hi' not in SR_REGISTRY or lfnorm is None or shfnorm is None:
    print('sr_hi not in registry or surface vars missing')
else:
    c = SR_REGISTRY['sr_hi']['constants']
    ah,bh,ch,dh = c['a'],c['b'],c['c'],c['d']
    Mhi = rhk[valid]; Ihi = thetaek[valid]+bh*thetaestark[valid]+ch
    T   = np.maximum(Mhi,Ihi)
    S   = np.maximum(lfnorm[valid],shfnorm[valid])
    P   = trueflat[valid]
    Tlo,Thi = prange(T); Slo,Shi = prange(S)
    latlim  = (float(lats.min()),float(lats.max()))
    lonlim  = (float(lons.min()),float(lons.max()))
    lfmean2d  = lfnorm.reshape(refshape).mean(axis=0)
    shfmean2d = shfnorm.reshape(refshape).mean(axis=0)
    maxmean2d = np.maximum(lfnorm,shfnorm).reshape(refshape).mean(axis=0)
    climlim   = float(np.nanmax(np.abs([lfmean2d,shfmean2d,maxmean2d])))
    supp = to_phys((ah*T)**3)-to_phys((ah*T+dh*S)**3)
    Tcen,suppmeans,_ = bin1d(T,supp)
    Ssw    = np.linspace(Slo,Shi,200)
    Tpcts  = [25,50,75,90]
    Tfixed = np.percentile(T,Tpcts)
    kwmap  = dict(coast=True,lonlim=lonlim,latlim=latlim,
                  latlines=[10,15,20],lonlines=[65,75,85],grid=False)
    kwsfc  = dict(cmap='ColdHot',vmin=-climlim,vmax=climlim)
    fig,axs = pplt.subplots([[1,1,2,2,3,3],[0,4,4,5,5,0]],figwidth=6.5,
                             proj={1:'cyl',2:'cyl',3:'cyl'},share=False,wspace=0.75,hspace=9)
    ms = axs[0].pcolormesh(lons,lats,lfmean2d,**kwsfc)
    axs[1].pcolormesh(lons,lats,shfmean2d,**kwsfc)
    axs[2].pcolormesh(lons,lats,maxmean2d,**kwsfc)
    axs[0].format(title=r'$\widetilde{\mathrm{LF}}$',latlabels='l',lonlabels='b',**kwmap)
    axs[1].format(title=r'$\widetilde{\mathrm{SHF}}$',lonlabels='b',**kwmap)
    axs[2].format(title='$\mathit{S}$',lonlabels='b',**kwmap)
    axs[3].scatter(Tcen,suppmeans,color=COLORS['sr_hi'],s=20,zorder=4)
    axs[3].axhline(0,color='gray',lw=0.5)
    axs[3].format(xlabel='$\mathit{T}$',xlim=(-1.75,1.25),xticks=0.5,
                  ylabel='Total Precipitation (mm)',ylim=(-0.1,3.1),yticks=1,
                  title='Suppression by $\mathit{\lambda S}$')
    for perc,tv,ls in zip(Tpcts,Tfixed,['-','--',':','-.']):
        axs[4].plot(Ssw,to_phys((ah*tv+dh*Ssw)**3),color=COLORS['sr_hi'],ls=ls,
                    label=fr'$\mathit{{T}}_{{{perc}}}$',lw=1)
    axs[4].axhline(0,color='gray',lw=0.5)
    axs[4].sharey(axs[3])
    axs[4].format(xlabel='$\mathit{\lambda S}$',xlim=(-0.5,1.75),xticks=0.4,
                  title='SR-HI at Fixed $\mathit{T}$')
    axs[4].tick_params(axis='y',labelleft=False)
    axs[4].legend(loc='ur',ncols=2,fontsize=7)
    fig.format(abc=True,titleloc='l',grid=False)
    fig.canvas.draw()
    figh  = fig.get_figheight()
    left  = axs[0].get_position().x0
    right = axs[2].get_position().x1
    btop  = min(axs[0].get_position().y0,axs[1].get_position().y0,axs[2].get_position().y0)
    tbot  = max(axs[3].get_position().y1,axs[4].get_position().y1)
    cax   = fig.add_axes([left, tbot+0.6*(btop-tbot)-0.09/figh, right-left, 0.18/figh])
    cb    = fig.colorbar(ms,cax=cax,orientation='horizontal',
                         label='Standardized Anomaly',extend='both')
    cb.ax.tick_params(labelsize=9)
    pplt.show()
    fig.save('../figs/fig_5.jpg')